In [7]:
pip install pyspark

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\arbaa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Creating Spark Session

This code block will allow the spark session to start and configure spark to load the spark-xml package in order to read the XML wiki file.

In [8]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("WikipediaWordCount")
    .config("spark.jar.packages", "com.databricks:spark-xml_2.12:0.18.0")
    .getOrCreate()
)

This code block creates a dataframe to read the xml file, then allows spark to block each '<page> ... </page>' in the file as a row in the dataframe. Lastly, the load function locates and loads the xml file from the data folder.

In [9]:
dataframe = (
    spark.read.format("xml")
    .option("rowTag", "page")
    .load("data\simplewiki-latest-pages-articles.xml\simplewiki-latest-pages-articles.xml")
)

In [14]:
spark

This selectExpr function allows me to display the sum of the pages in one colum under the name total_pages.

I was also able to show the structure of the dataframe by printing out the schema. This gives me a clearer idea of the layout of the data within the XML file.

In [10]:
dataframe.selectExpr("count(1) as total_pages").show()
dataframe.printSchema()

+-----------+
|total_pages|
+-----------+
|     536723|
+-----------+

root
 |-- id: long (nullable = true)
 |-- ns: long (nullable = true)
 |-- redirect: struct (nullable = true)
 |    |-- _title: string (nullable = true)
 |-- revision: struct (nullable = true)
 |    |-- comment: string (nullable = true)
 |    |-- contributor: struct (nullable = true)
 |    |    |-- _deleted: string (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- ip: string (nullable = true)
 |    |    |-- username: string (nullable = true)
 |    |-- format: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- minor: string (nullable = true)
 |    |-- model: string (nullable = true)
 |    |-- origin: long (nullable = true)
 |    |-- parentid: long (nullable = true)
 |    |-- sha1: string (nullable = true)
 |    |-- text: struct (nullable = true)
 |    |    |-- _VALUE: string (nullable = true)
 |    |    |-- _bytes: long (nullable = true)
 |    |    |-- _sha1: string (nulla

## Testing with a smaller sample of data

Creating a variable pages to hold the dataframe with selected columns, then displaying 5 rows to verify the data.


In [13]:
from pyspark.sql import functions as F

pages = (
    dataframe.select(
        F.col("title"),
        F.col("revision.text._VALUE").alias("text")
    )
    .filter(F.col("text").isNotNull())
)

In [27]:
pages

DataFrame[title: string, text: string]

In [31]:
#pages.count()
#pages.show(5, truncate=90)

cleanText = (
    pages
    .withColumn("cleanText", F.lower(F.col("text"))) # convert to lowercase
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"\{\{[^}]*\}\}", " ")) # remove templates {{...}}
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"\[\[([^\]|]*\|)?([^\]]*)\]\]", "$2")) # remove wiki links [[...]]
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"<[^>]+>", " ")) # remove HTML tags
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"<ref[^>]*>.*?</ref>", " ")) # remove references <ref>...</ref>
    .withColumn("cleanText", F.regexp_replace("cleanText", r"[^a-z\s]", " ")) # make sure only letters and spaces remain
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"\s+", " ")) # replace multiple spaces with a single space
    .withColumn("cleanText", F.trim(F.col("cleanText"))) # remove any space from the beginning and end
    .filter(F.length(F.col("cleanText")) > 0) # filter out empty texts
)

cleanText.show(5, truncate=90)

+------+------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------+
| title|                                                                                      text|                                                                                 cleanText|
+------+------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------+
| April|{{monththisyear|4}}\n'''April''' (Apr.) is the fourth [[month]] of the [[year]] in the ...|april apr is the fourth month of the year in the julian and gregorian calendars and com...|
|August|{{monththisyear|8}}\n'''August''' (Aug.) is the eighth [[month]] of the [[year]] in the...|august aug is the eighth month of the year in the gregorian calendar coming between jul...|
|   Art|[[File:Pierre-Auguste_Renoir,_Le_Moul